In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

TRAIN_PATH = "Dataset/train"
TEST_PATH = "Dataset/test"
MODEL_PATH = "Model/model.xml"

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

In [2]:
def preprocessing(img):
    img = cv2.resize(img, (100,100))
    img = cv2.equalizeHist(img)
    return img

def detect_face(img):
    faces = face_cascade.detectMultiScale(img, 1.05, 2, minSize=(30,30))
    if len(faces) > 0:
        x,y,w,h = faces[0]
        return preprocessing(img[y:y+h, x:x+w]), (x,y,w,h)
    
    w,h = img.shape
    return preprocessing(img), (0,0,w,h)

In [3]:
def load_data(folder_path, class_names):
    faces, labels = [], []
    for label, class_name in enumerate(class_names):
        class_path = os.path.join(folder_path, class_name)
        if not os.path.dirname(class_path):
            continue

        for filename in sorted(os.listdir(class_path)):
            img = cv2.imread(os.path.join(class_path, filename), cv2.IMREAD_GRAYSCALE)
            face_img, _ = detect_face(img)
            faces.append(face_img)
            labels.append(label)

    return faces, np.array(labels, dtype=np.int32)

In [4]:
def train_test_model():
    class_names = sorted(os.listdir(TRAIN_PATH))

    print("Load Training Data...")
    train_faces, train_labels = load_data(TRAIN_PATH, class_names)
    if train_faces is None:
        print("Training Data Not Found!")
        return

    face_recognition = cv2.face.LBPHFaceRecognizer_create()
    face_recognition.train(train_faces, train_labels)

    print("Load Test Data...")
    test_faces, test_labels = load_data(TEST_PATH, class_names)
    if test_faces is None:
        print("Test Data Not Found!")
        return

    counter = 0 
    for face, true_predict in zip(test_faces, test_labels):
        predicted_label, confidence = face_recognition.predict(face)
        counter += predicted_label == true_predict
        status = "RIGHT" if predicted_label == true_predict else "WRONG"

        print(f"actual: {true_predict} | predict: {predicted_label} | confidence: {confidence:.2f} | {status}")

    print(f"Average Accuracy: {counter/len(test_faces) * 100:.2f}%")

    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    face_recognition.write(MODEL_PATH)
    print("Model Save Successfully")

In [ ]:
def predict(img_path):
    if os.path.exists(MODEL_PATH):
        print("Model Not Found. Please Train Model First!")
        return
    
    class_names = sorted(os.listdir(TRAIN_PATH))
    face_recognition = cv2.face.LBPHFaceRecognizer_create
    face_recognition.read(MODEL_PATH)

    img = cv2.imread(img_path)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    face_img, (x,y,w,h) = detect_face(img_gray)
    predict_label, confidence = face_recognition(face_img)
    predict_class = class_names[predict_label]

    cv2.rectangle(img, (x,y), (x+w, y+h), (255,0,0), 1)
    cv2.putText(img, f"{predict_class} | {confidence:.2f}", (x, y-10), cv2.FONT_HERSHEY_PLAIN, 1, (0,255,0))

    print(f"{img_path} detected as {predict_class}")
    print(f"")